In [6]:
import pandas as pd
import random
from datetime import datetime, timedelta

# ==========================================
# 1. BACA DATASET KAGGLE ASLI
# ==========================================
# Pastikan nama file sesuai dengan yang Anda download (misal: 'healthcareTest.csv')
nama_file_kaggle = 'healthcareTest.csv' 

try:
    df_kaggle = pd.read_csv(nama_file_kaggle)
    # Memastikan patIndex di dataset Kaggle bertipe int64
    df_kaggle['patIndex'] = df_kaggle['patIndex'].astype('int64')
    
    # Ambil daftar ID pasien asli (berupa angka)
    daftar_pasien_asli = df_kaggle['patIndex'].unique().tolist()
    print(f"✅ Berhasil membaca dataset Kaggle. Ditemukan {len(daftar_pasien_asli)} pasien.")
except FileNotFoundError:
    print(f"❌ File '{nama_file_kaggle}' tidak ditemukan. Pastikan file ada di folder yang sama!")
    exit()

# ==========================================
# 2. GENERATE DATA TINDAKAN MEDIS (DUMMY)
# ==========================================
referensi_tindakan = [
    ("CPT-80053", "Laboratory", "Comprehensive Metabolic Panel", 50.00),
    ("CPT-71045", "Radiology", "Chest X-Ray, single view", 120.00),
    ("CPT-70450", "Radiology", "CT Scan, Head/Brain", 850.00),
    ("ICD10-0DTJ", "Surgery", "Appendectomy (Open)", 8500.00),
    ("CPT-93000", "Diagnostic", "Electrocardiogram (ECG)", 75.00)
]

def generate_tanggal_acak():
    start_date = datetime(2025, 1, 1)
    end_date = datetime(2026, 5, 24)
    selisih_hari = (end_date - start_date).days
    return start_date + timedelta(days=random.randint(0, selisih_hari))

# Kita buat total 1000 records tindakan medis acak untuk pasien-pasien tersebut
records = []
for i in range(1, 1001):
    tindakan = random.choice(referensi_tindakan)
    cost = round(tindakan[3] * random.uniform(0.9, 1.1), 2)
    
    records.append({
        "Procedure_ID": f"TR-{1000 + i}",
        "patIndex": random.choice(daftar_pasien_asli), # Menggunakan ID angka asli dari Kaggle
        "Procedure_Date": generate_tanggal_acak().strftime('%Y-%m-%d'),
        "Procedure_Code": tindakan[0],
        "Procedure_Category": tindakan[1],
        "Procedure_Description": tindakan[2],
        "Procedure_Cost": cost
    })

df_procedures = pd.DataFrame(records)
# Memastikan patIndex di tabel dummy juga bertipe int64 agar klop saat di-merge
df_procedures['patIndex'] = df_procedures['patIndex'].astype('int64')

# Simpan tabel tindakan medis terpisah (opsional jika Anda butuh file mentahnya)
df_procedures.to_csv('medical_procedures_corrected.csv', index=False)

# ==========================================
# 3. PROSES PENGGABUNGAN (MERGE)
# ==========================================
print("⏳ Sedang melakukan proses penggabungan data...")

# Lakukan agregasi agar formatnya tetap 1 baris = 1 pasien (Cocok untuk ML)
df_agg = df_procedures.groupby('patIndex').agg(
    Total_Procedures=('Procedure_ID', 'count'),
    Total_Procedure_Cost=('Procedure_Cost', 'sum')
).reset_index()

# Sekarang merge dijamin aman karena kedua 'patIndex' sudah bertipe int64
df_final = pd.merge(df_kaggle, df_agg, on='patIndex', how='left')

# Mengisi pasien yang tidak mendapat tindakan medis dummy dengan angka 0
df_final['Total_Procedures'] = df_final['Total_Procedures'].fillna(0).astype(int)
df_final['Total_Procedure_Cost'] = df_final['Total_Procedure_Cost'].fillna(0)

# ==========================================
# 4. EXPORT HASIL AKHIR
# ==========================================
nama_output = 'healthcare_dataset_complete.csv'
df_final.to_csv(nama_output, index=False)

print(f"🎉 Sukses! File '{nama_output}' berhasil dibuat tanpa error tipe data.")

✅ Berhasil membaca dataset Kaggle. Ditemukan 344 pasien.
⏳ Sedang melakukan proses penggabungan data...
🎉 Sukses! File 'healthcare_dataset_complete.csv' berhasil dibuat tanpa error tipe data.


In [10]:
df = pd.read_csv('healthcareTest.csv')
df

,patIndex,pdc,num_ip_post,total_los_post,num_op_post,num_er_post,num_ndc_post,num_gpi6_post,adjust_total_30d_post,generic_rate_post,...,brand_cost,ratio_G_total_cost,numofgen_post,numofbrand_post,generic_cost_post,brand_cost_post,ratio_G_total_cost_post,pdc_80_flag,drug_class,patient_key
0,2,0.333333,0,0,4,0,15,5,14.466667,0.101382,...,2984.927229,0.010155,2,13,196.359216,3001.501507,0.061403,0,*ANTIDIABETICS*,168
1,5,0.866667,0,0,5,0,16,4,18.000000,0.888889,...,0.000000,1.000000,14,2,671.755173,735.661568,0.477297,1,*ANTIDIABETICS*,499
2,21,0.500000,0,0,0,0,8,6,8.000000,0.875000,...,0.000000,1.000000,7,1,50.160767,41.220633,0.548917,0,*ANTIDIABETICS*,1830
3,22,0.977778,0,0,9,0,40,9,42.533333,0.835423,...,1345.104492,0.339094,33,7,842.908516,1695.649323,0.332042,1,*ANTIDIABETICS*,1852
4,33,0.527778,0,0,6,0,28,7,28.000000,0.964286,...,0.000000,1.000000,27,1,1163.290225,6.514435,0.994431,0,*ANTIDIABETICS*,3369
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
339,3488,0.827778,1,4,3,1,14,9,16.033333,0.833680,...,0.000000,NaN,11,3,133.680794,135.766288,0.496130,1,*ANTIDIABETICS*,510279
340,3495,0.888889,2,30,35,1,88,16,83.466667,0.533147,...,734.085933,0.190072,47,41,911.805080,4649.972588,0.163941,1,*ANTIDIABETICS*,512465
341,3511,0.888889,0,0,19,0,6,5,15.133333,0.405286,...,237.477793,0.531823,3,3,131.855141,232.932311,0.361457,1,*ANTIDIABETICS*,516602
342,3520,0.177778,0,0,2,0,3,1,1.000000,1.000000,...,0.000000,1.000000,3,0,74.389252,0.000000,1.000000,0,*ANTIDIABETICS*,517252


In [13]:
df = pd.read_csv("healthcare_joined_10000.csv")
df

,patIndex,pdc,num_ip_post,total_los_post,num_op_post,num_er_post,num_ndc_post,num_gpi6_post,adjust_total_30d_post,generic_rate_post,...,ratio_G_total_cost_post,pdc_80_flag,drug_class,patient_key,Procedure_ID,Procedure_Date,Procedure_Code,Procedure_Category,Procedure_Description,Cost
0,7736,0.652089,0,2,0,1,21,3,39.362393,0.589971,...,0.290760,0,*ANTIDIABETICS*,564447,TR-27736,2025-06-05,ICD10-0DTJ,Surgery,Appendectomy (Open),7677.29
1,6796,0.126829,0,1,1,0,29,10,1.000000,1.000000,...,NaN,0,*ANTIDIABETICS*,128808,TR-26796,2025-07-26,ICD10-0DTJ,Surgery,Appendectomy (Open),7939.83
2,6601,0.071520,0,0,5,0,14,1,19.743287,0.898207,...,NaN,0,*ANTIDIABETICS*,468260,TR-26601,2025-09-06,CPT-70450,Radiology,"CT Scan, Head/Brain",848.62
3,8545,0.193326,1,0,9,0,4,3,22.062451,0.245129,...,1.000000,0,*ANTIDIABETICS*,481476,TR-28545,2025-03-19,CPT-70450,Radiology,"CT Scan, Head/Brain",835.70
4,5707,1.000000,0,4,17,1,8,10,6.229274,0.682387,...,0.000000,1,*ANTIDIABETICS*,218875,TR-25707,2025-06-08,CPT-93000,Diagnostic,Electrocardiogram (ECG),70.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,6402,0.800000,1,3,10,1,8,9,43.708230,0.500306,...,0.045707,1,*ANTIDIABETICS*,12272,TR-26402,2025-08-13,CPT-80053,Laboratory,Comprehensive Metabolic Panel,47.12
9996,3727,0.800000,0,3,0,0,1,7,38.100976,0.674009,...,0.989839,1,*ANTIDIABETICS*,262228,TR-23727,2025-05-22,CPT-70450,Radiology,"CT Scan, Head/Brain",893.34
9997,9003,0.539992,0,0,9,0,15,8,1.000000,0.537978,...,1.000000,0,*ANTIDIABETICS*,478540,TR-29003,2025-12-05,CPT-80053,Laboratory,Comprehensive Metabolic Panel,45.26
9998,4638,0.829381,0,0,0,1,10,10,24.796935,1.000000,...,NaN,1,*ANTIDIABETICS*,68135,TR-24638,2025-06-04,ICD10-0DTJ,Surgery,Appendectomy (Open),7921.52
